# 4. Warum gestapelte lineare Schichten nichts bringenAus *Transformer von Grund auf*.

In [ ]:
# In Colab liegt nichts auf der Festplatte, also holt diese Zelle# die zwei oder drei Dateien, die das Kapitel braucht. Wer das# Repository geklont hat, findet sie schon vor, und es wird# nichts heruntergeladen.import osimport sysimport urllib.requestBASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/notebooks/de/code/"PLACES = [".", "code", "../code"]for name in ["ch04_nonlinearity.py"]:    if not any(os.path.exists(os.path.join(where, name))               for where in PLACES):        urllib.request.urlretrieve(BASE + name, name)for where in PLACES:    if os.path.isdir(where) and where not in sys.path:        sys.path.insert(0, where)

Warum ein Stapel linearer Schichten eine Schicht ist.Zwei Matrixmultiplikationen hintereinander lassen sich immer durch eine einzige Matrix ersetzen. Diese Datei beweist das mit Zahlen, statt es zu behaupten, und misst dann, was die Aktivierungsfunktionen dagegen tun.

In [ ]:
import numpy as npnp.set_printoptions(precision=3, suppress=True, floatmode="fixed")W1 = np.array([[0.5, -1.0, 2.0],               [1.5, 0.0, -0.5]])W2 = np.array([[1.0, 0.0],               [2.0, -1.0],               [0.0, 1.0]])X = np.array([[1.0, 2.0],              [3.0, -1.0]])

In [ ]:
def relu(z):    return np.maximum(z, 0.0)

Die tanh-Näherung, die Implementierungen tatsächlich benutzen.

In [ ]:
def gelu(z):    inner = np.sqrt(2.0 / np.pi) * (z + 0.044715 * z ** 3)    return 0.5 * z * (1.0 + np.tanh(inner))

In [ ]:
def silu(z):    return z / (1.0 + np.exp(-z))

Zwei Schichten ohne Aktivierung, gegen eine gleichwertige.

In [ ]:
def collapse_demo():    two_steps = (X @ W1) @ W2    combined = W1 @ W2    one_step = X @ combined    return two_steps, combined, one_step

Wie viele ReLU-Einheiten eine Zufallseingabe abschaltet.

In [ ]:
def dead_units(seed=0, n=4000, d=64):    rng = np.random.default_rng(seed)    x = rng.standard_normal((n, d))    w = rng.standard_normal((d, d)) / np.sqrt(d)    for bias, label in [(0.0, "Bias 0.0"), (-1.0, "Bias -1.0")]:        activated = relu(x @ w + bias)        share = float((activated == 0.0).mean())        yield label, share

In [ ]:
def main():    two_steps, combined, one_step = collapse_demo()    print("ZWEI LINEARE SCHICHTEN, KEINE AKTIVIERUNG")    print("  x durch W1, dann durch W2")    print(two_steps)    print("  die eine Matrix W1 @ W2, Form", combined.shape)    print(combined)    print("  x durch diese eine Matrix")    print(one_step)    print("  identisch:", np.allclose(two_steps, one_step))    print("\nDIESELBEN ZWEI SCHICHTEN MIT EINER RELU DAZWISCHEN")    bent = relu(X @ W1) @ W2    print(bent)    print("  keine einzelne Matrix bildet das nach, und darum geht es")    print("\nDREI AKTIVIERUNGSFUNKTIONEN AN FÜNF EINGABEN")    z = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])    print("  Eingabe", z)    print("  relu   ", relu(z))    print("  gelu   ", gelu(z))    print("  silu   ", silu(z))    print("\nWIE VIELE RELU-EINHEITEN BEI NULL LIEGEN")    for label, share in dead_units():        print(f"  {label:<12} {100 * share:.1f} Prozent abgeschaltet")

In [ ]:
main()

Das Kapitel, aus dem dieses Notebook stammt, rechnet jede Zahl oben durch und erklärt, warum jeder Schritt da ist.